# Projet Home Credit – Sujet 03

*Séance :* 01  
*Nom :* AKPOTIN Fresnel  
*Porteur :* P3


*
É
t
a
p
e
s
 
r
é
a
l
i
s
é
e
s
 
:
*

-
 
1
.
4
 
:
 
c
a
r
d
i
n
a
l
i
t
é

-
 
1
.
6
 
:
 
t
r
o
u
s

-
 
1
.
9
 
:
 
c
i
b
l
e

-
 
1
.
1
2
 
:
 
c
o
û
t
 
d
e
s
 
e
r
r
e
u
r
s

In [8]:
import pandas as pd

train = pd.read_csv("train.csv", low_memory=False)
test = pd.read_csv("a_decider.csv", low_memory=False)
print(train.shape, test.shape)

(246008, 122) (61503, 121)


## 1.4 — Cardinalité

In [9]:
# nombre de valeurs distinctes par colonne
distinctes = train.nunique().sort_values()
print(distinctes.to_string())

FLAG_MOBIL                           1
TARGET                               2
FLAG_OWN_REALTY                      2
NAME_CONTRACT_TYPE                   2
FLAG_WORK_PHONE                      2
FLAG_CONT_MOBILE                     2
FLAG_EMP_PHONE                       2
FLAG_OWN_CAR                         2
FLAG_PHONE                           2
FLAG_EMAIL                           2
LIVE_CITY_NOT_WORK_CITY              2
REG_CITY_NOT_WORK_CITY               2
LIVE_REGION_NOT_WORK_REGION          2
REG_CITY_NOT_LIVE_CITY               2
REG_REGION_NOT_LIVE_REGION           2
REG_REGION_NOT_WORK_REGION           2
FLAG_DOCUMENT_18                     2
FLAG_DOCUMENT_19                     2
FLAG_DOCUMENT_20                     2
FLAG_DOCUMENT_21                     2
FLAG_DOCUMENT_6                      2
FLAG_DOCUMENT_7                      2
FLAG_DOCUMENT_8                      2
FLAG_DOCUMENT_9                      2
FLAG_DOCUMENT_10                     2
FLAG_DOCUMENT_11         

In [10]:
# les colonnes texte et leur nombre de catégories
texte = train.select_dtypes(exclude="number").columns
print(train[texte].nunique().sort_values(ascending=False))

ORGANIZATION_TYPE             58
OCCUPATION_TYPE               18
NAME_INCOME_TYPE               8
NAME_TYPE_SUITE                7
WALLSMATERIAL_MODE             7
WEEKDAY_APPR_PROCESS_START     7
NAME_FAMILY_STATUS             6
NAME_HOUSING_TYPE              6
NAME_EDUCATION_TYPE            5
FONDKAPREMONT_MODE             4
HOUSETYPE_MODE                 3
CODE_GENDER                    3
FLAG_OWN_CAR                   2
NAME_CONTRACT_TYPE             2
FLAG_OWN_REALTY                2
EMERGENCYSTATE_MODE            2
dtype: int64


**Constats**
- `FLAG_MOBIL` n'a qu'une seule valeur → inutile, on la retire.
- 16 colonnes texte ; la plus variée est `ORGANIZATION_TYPE` avec **58 catégories**, dont beaucoup très rares → à regrouper.

## 1.6 — Trous

In [11]:
# pourcentage de valeurs manquantes par colonne
manquants = train.isna().mean() * 100
manquants = manquants[manquants > 0].sort_values(ascending=False)
print(len(manquants), "colonnes avec des trous,", (manquants > 30).sum(), "à plus de 30 %")
print(manquants.round(2).to_string())

66 colonnes avec des trous, 50 à plus de 30 %
COMMONAREA_MODE                 69.88
COMMONAREA_AVG                  69.88
COMMONAREA_MEDI                 69.88
NONLIVINGAPARTMENTS_MEDI        69.47
NONLIVINGAPARTMENTS_MODE        69.47
NONLIVINGAPARTMENTS_AVG         69.47
LIVINGAPARTMENTS_MEDI           68.39
LIVINGAPARTMENTS_MODE           68.39
LIVINGAPARTMENTS_AVG            68.39
FONDKAPREMONT_MODE              68.38
FLOORSMIN_MODE                  67.85
FLOORSMIN_AVG                   67.85
FLOORSMIN_MEDI                  67.85
YEARS_BUILD_MEDI                66.50
YEARS_BUILD_MODE                66.50
YEARS_BUILD_AVG                 66.50
OWN_CAR_AGE                     65.98
LANDAREA_MEDI                   59.35
LANDAREA_MODE                   59.35
LANDAREA_AVG                    59.35
BASEMENTAREA_AVG                58.47
BASEMENTAREA_MODE               58.47
BASEMENTAREA_MEDI               58.47
EXT_SOURCE_1                    56.39
NONLIVINGAREA_MODE              55.16
NONL

In [12]:
# le taux de défaut change-t-il quand la valeur manque ?
for col in ["EXT_SOURCE_1", "EXT_SOURCE_3", "OCCUPATION_TYPE", "AMT_REQ_CREDIT_BUREAU_YEAR"]:
    vide = train[col].isna()
    print(f"{col:28} vide : {train.TARGET[vide].mean():.1%}   rempli : {train.TARGET[~vide].mean():.1%}")

EXT_SOURCE_1                 vide : 8.5%   rempli : 7.5%
EXT_SOURCE_3                 vide : 9.3%   rempli : 7.8%
OCCUPATION_TYPE              vide : 6.5%   rempli : 8.8%
AMT_REQ_CREDIT_BUREAU_YEAR   vide : 10.4%   rempli : 7.7%


**Constats**
- **66 colonnes** ont des trous, **50 à plus de 30 %** (jusqu'à 69,9 %), surtout les colonnes sur le logement.
- Le trou est une information : quand `EXT_SOURCE_3` manque, le défaut passe de 7,8 % à **9,3 %**.
  → on garde une colonne « était manquant » en plus de remplir la valeur.

## 1.9 — Cible

In [13]:
# répartition de la cible
print(train.TARGET.value_counts())
print(f"taux de défaut : {train.TARGET.mean():.2%}")

TARGET
0    226148
1     19860
Name: count, dtype: int64
taux de défaut : 8.07%


In [14]:
# taux de défaut selon quelques variables
print(train.groupby("NAME_EDUCATION_TYPE").TARGET.mean().sort_values())
print(train.groupby("CODE_GENDER").TARGET.mean())

NAME_EDUCATION_TYPE
Academic degree                  0.007752
Higher education                 0.053726
Incomplete higher                0.083192
Secondary / secondary special    0.089361
Lower secondary                  0.111767
Name: TARGET, dtype: float64
CODE_GENDER
F      0.069649
M      0.102143
XNA    0.000000
Name: TARGET, dtype: float64


**Constats**
- **8,07 % de défauts** (19 860 sur 246 008) : la cible est déséquilibrée.
- Un modèle qui dit « remboursé » à tout le monde a 91,9 % de bonnes réponses mais ne trouve **aucun** défaut → l'exactitude n'est pas une bonne métrique ici.
- Le défaut varie selon le profil : 5,4 % avec un diplôme supérieur contre 11,2 % pour *Lower secondary*.

## 1.12 — Coût des erreurs

Deux erreurs possibles :
- **accorder un crédit qui finit en défaut** → on perd une partie de l'argent prêté ;
- **refuser un bon client** → on perd seulement les intérêts qu'il aurait payés.

La première coûte beaucoup plus cher. On fixe : **un défaut accordé coûte 10, un bon client refusé coûte 1**.

In [15]:
COUT_DEFAUT_ACCORDE = 10
COUT_BON_CLIENT_REFUSE = 1

nb_defauts = train.TARGET.sum()
nb_bons = (train.TARGET == 0).sum()

# deux décisions sans modèle
cout_tout_accorder = nb_defauts * COUT_DEFAUT_ACCORDE
cout_tout_refuser = nb_bons * COUT_BON_CLIENT_REFUSE
print("tout accorder :", cout_tout_accorder)
print("tout refuser  :", cout_tout_refuser)

tout accorder : 198600
tout refuser  : 226148


**Constats**
- Tout accorder coûte **198 600**, tout refuser coûte **226 148**.
- Le modèle doit faire moins que **198 600** pour servir à quelque chose.
- Avec ce rapport de 10 pour 1, refuser un client vaut le coup dès que son risque de défaut dépasse **1 / (1 + 10) ≈ 9 %**. Ce sera notre point de départ pour le seuil.